# Profils des enquêtés

## Les pratiques de travail

In [1]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

last_file="anonymous_answer_2026-07-24.csv"


In [3]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])]

PermissionError: Forbidden

In [4]:

df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")


In [5]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [6]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";",sep) ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [7]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [8]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

In [9]:
!python -m spacy download fr_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/45.8 MB ? eta -:--:--

     ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/45.8 MB 91.4 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 32.0/45.8 MB 88.0 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 79.6 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [10]:
import spacy
import re
import bertopic

import nltk
from nltk.corpus import stopwords

In [11]:
nlp = spacy.load('fr_core_news_md')
nlp.add_pipe("merge_entities")
#nlp.add_pipe("merge_noun_chunks")
spacy_stopwords = list(nlp.Defaults.stop_words)




nltk.download('stopwords')

spacy_stopwords
stop_words = set(stopwords.words('french'))


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
#
pos=['VERB','ADJ','ADV']
df_ia = df0[["q45_clé","q35_ia_tools","q37_raison_util_ia"]].loc[~df0.q37_raison_util_ia.isna()]
dict_sent = {}
for n, x in enumerate(df_ia.q35_ia_tools):
    doc = nlp(x)
    if len([t.text.lower() for t in doc if t.pos_ in pos]) >=2:
        print(x)
        id_user = df_ia.q45_clé.iloc[n]
        dict_sent[id_user] = x
    

NameError: name 'df0' is not defined

In [13]:
dict_new_sent = {}
for n, x in enumerate(df_ia.q37_raison_util_ia):
    id_user= df_ia.q45_clé.iloc[n]
    if id_user in dict_sent.keys():
        new_sent = ". ".join([dict_sent[id_user], x])
    else:
        new_sent = x

    dict_new_sent[id_user]= new_sent

len(dict_new_sent)


NameError: name 'df_ia' is not defined

## Preprocess text

In [14]:
df_ia["text"]=df_ia.q45_clé.map(dict_new_sent.get)

NameError: name 'df_ia' is not defined

In [15]:
df_ia["clean_text"] = df_ia.apply(lambda row: str(row.text).lower().strip(), 1) # lower sentence
df_ia["clean_text"] = df_ia.apply(lambda row: re.sub(r"\w'", "", str(row.clean_text)), 1) #replace "\w’" by ""
df_ia["clean_text"] = df_ia.apply(lambda row: re.sub(r"\/", " ", str(row.clean_text)), 1) #replace "/" by " "
df_ia["clean_text"] = df_ia.apply(lambda row: re.sub(r"\(|\)", "", str(row.clean_text)), 1) #replace "/" by " "
df_ia["clean_text"] = df_ia.apply(lambda row: re.sub(r"\+", ".", str(row.clean_text)), 1) #replace "/" by " "
df_ia["clean_text"] = df_ia.apply(lambda row: re.sub(r"œ", "oe", str(row.clean_text)), 1) #replace "/" by " "
df_ia["split_text"] = df_ia.clean_text.str.split(".")

######### Explode

df_ia1 = df_ia.explode("split_text")
df_ia1 = df_ia1.loc[df_ia1.split_text.str.len()>2]
df_ia1["split_text"] = df_ia1.apply(lambda row: " ".join(re.sub("[^'a-zA-Zàâäéèêëïîôöùûüÿçß]+", " ", row.split_text).split()), 1) #remove hashtag, arobase, HTML character
df_ia1["tokens"] = df_ia1.apply(lambda row: row.split_text.split(),1) 


NameError: name 'df_ia' is not defined

In [16]:
df_ia1["len_txt"] = df_ia1.tokens.str.len()
df_ia1 = df_ia1.loc[df_ia1.tokens.str.len()>2]

print(f"Il y a {len(df_ia1)} phrases. Le plus long fait {max(df_ia1.len_txt)} caractères.")
df_ia1

NameError: name 'df_ia1' is not defined

In [17]:
df_ia1["text_id"] = "id_" + df_ia1.index.astype(str) # on crée un id qu'on réutilisera plus bas
df_ia1
texts = [text for text in df_ia1.split_text] #liste des textes nettoyés
texts_id = [id_text for id_text in df_ia1.text_id]

NameError: name 'df_ia1' is not defined

## Décomposition de l'algorithme Bertopic

BERTopic est composé de quatre "étapes" :

   - Le plongement dans un embedding des phrases du corpus à l'aide d'un modèle sentence transformers
   - La réduction de la dimensionalité de l'embedding
   - L'extraction de clusters
   - La labellisation des clusters en utilisant c-TF-IDF

Le point essentiel à retenir est que chacune des fonctions utilisées pour réaliser ces 4 étapes possèdent des paramètres dont la définition joue ensuite sur le résultat final.

In [ ]:
from sentence_transformers import SentenceTransformer
import pickle



La librairie sentence_transformers permet d'accéder facilement aux modèles hébergés sur huggingface.

pickle servira à enregistrer l'embedding pour une réutilisation ultérieure.

Une fois le modèle chargé, il peut être utile de vérifier que la taille maximale de la séquence admise par le modèle est égale ou supérieure à la taille du texte le plus long du corpus. Dans le cas contraire, les textes trop longs seront tronqués.

Le modèle que nous utilisons est 'paraphrase-multilingual-MiniLM-L12-v2'. La taille maximale de la séquence est 128, or le commentaire le plus long est 63 mots.


In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Taille maximale de la séquence : ", model.get_max_seq_length())
print(model.get_max_seq_length()>= max(df_ia1.len_txt))


In [ ]:
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

### Réduction des dimensions avec UMAP



Une fois l’embedding réalisée, BERTopic fait appel à UMAP (Uniform Manifold Approximation and Projection) pour réduire le nombre de dimensions – qui est au départ de 384.

UMAP est un algorithme qui cherche à préserver la structure locale (relation entre points proches) et la structure globale des données.

    Le nombre de voisins : n_neighbors.

La balance entre la préservation des structures locales et globales est d'abord déterminée par le nombre de voisins (n_neighbors) de chaque point que l'algorithme prend en compte. Plus le n_neighbors est petit, plus UMAP préservera la structure locale des données. Inversement, plus n_neighbors est grand, plus UMAP rendra visible la structure globale au détriment des spécifités locales.

Pour saisir l'effet de n_neighbors, on peut s'amuser à tracer une série de scatterplots.


In [ ]:
import umap
from tqdm.auto import tqdm

In [ ]:
embeddings.shape

In [ ]:


fig, ax = plt.subplots(3, 3, figsize=(14, 14))
nns = [2, 3, 4, 5, 10, 15, 20, 30, 100]
i, j = 0, 0
for n_neighbors in tqdm(nns):
    fit = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.05, n_components=2, random_state=42, metric = 'cosine')
    u = fit.fit_transform(embeddings)
    sns.scatterplot(x=u[:,0], y=u[:,1],  ax=ax[j, i])
    ax[j, i].set_title(f'n={n_neighbors}')
    if i < 2: i += 1
    else: i = 0; j += 1



Visuellement, il semble que le nombre de voisin optimal se situe entre 2 et 4. On distingue plus facilement des grappes.

In [ ]:
## parametre umap
n_neighbors=4
n_components=5
min_dist=0.00

##parametre hdbscan
min_samples=2
min_cluster_size=5

umap_model = umap.UMAP(n_neighbors=n_neighbors, 
                       n_components=n_components, 
                       min_dist=min_dist,
                       metric='cosine',
                       random_state=42).fit_transform(embeddings)

### Clusterisation avec Hdbscan

In [ ]:
import hdbscan

In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=min_samples,
    min_cluster_size=min_cluster_size,
    metric='euclidean',cluster_selection_method='eom', 
    gen_min_span_tree=True
).fit(umap_model)


In [ ]:
labels.single_linkage_tree_.plot()

In [ ]:
labels.labels_

In [ ]:
len(set(labels.labels_))

In [ ]:
dict_clusters = {}
for n, x in enumerate(texts):
    dict_clusters[x] = labels.labels_[n]


for x in range(-1,len(set(labels.labels_))):
    print("Cluster :", x)
    for v in dict_clusters:
        if dict_clusters[v] == x:
            print(v)

In [ ]:
umap_model = umap.UMAP(n_neighbors=n_neighbors, 
                       n_components=n_components, 
                       min_dist=min_dist,
                       metric='cosine',
                       random_state=42)

hdbscan_model =  hdbscan.HDBSCAN(
    min_samples=min_samples,
    min_cluster_size=min_cluster_size,
    metric='euclidean',
    cluster_selection_method='eom', 
    prediction_data = True,
    gen_min_span_tree=True
)



In [ ]:
from bertopic.vectorizers import ClassTfidfTransformer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
vectorizer_model = CountVectorizer(ngram_range=(1, 2),
                                   strip_accents='unicode',
                                    stop_words= [x for x in nlp.Defaults.stop_words])

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)


In [ ]:


topic_model = bertopic.BERTopic(
    language="french",
    #embedding_model= model,    # Step 1 - Extract embeddings
    umap_model=umap_model,              # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,        # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,  # Step 4 - Tokenize topics
    ctfidf_model=ctfidf_model,          # Step 5 - Extract topic words
    calculate_probabilities=True,        
    verbose=True
)



In [ ]:
topics, probs = topic_model.fit_transform(texts, embeddings)

In [ ]:
len(set(topics))

In [ ]:
top_lab = topic_model.get_topic_info()
top_lab

In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=46, n_words = 10)
fig

## Matrice de similarité et arbre hiérarchique

On regarde quels sont les topics proches et s'il peut être judicieux d'opérer des regroupements.

In [ ]:
topic_model.visualize_heatmap()

S'il y a des topics à regrouper, on utilisera les lignes de codes ci-dessous (voir la documentation de Bertopic sur [les moyens de réduire les topics](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html#manual-topic-reduction)):

```
topics_to_merge = [1, 2]
topic_model.merge_topics(docs, topics_to_merge)
```

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(texts)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

## Labellisation manuelle des topics

In [ ]:
topic_model.set_topic_labels({0:"Traduction", 1:"Comprendre l'IA", 2:"Relecture", 3:"Gain de temps", 4:"Traitement de données", 5:"Retranscription"})
topic_model.get_topic_info()

# Visualisation des documents
On réduit au préalable l'embedding à 2 dimensions avec Umap. C'est plus rapide à produire et correspond à une visualisation en 2D.

In [ ]:
# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = umap.UMAP(n_neighbors=4, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(texts, reduced_embeddings=reduced_embeddings)



In [ ]:
# with the original embeddings
fig = topic_model.visualize_document_datamap(texts, reduced_embeddings=reduced_embeddings,title=f"Les {len(set(topics))} raisons d'utiliser l'IA", custom_labels=True, datamap_kwds={"label_font_size":12})
#fig.savefig("../script_analyse/stat_descriptive/viz/raison_ia_bertopic.png", bbox_inches="tight")

In [ ]:
fig = topic_model.visualize_document_datamap(docs, reduced_embeddings=reduced_embeddings, interactive=True)

# Attribution des topics aux documents

In [ ]:
# 1. Récupération des infos sur les documents avec .get_document_info(). On enlève les colonne superflues
dftexts = topic_model.get_document_info(texts).sort_values(by=["Topic","Probability"], ascending=[True, False])#.drop(columns=["CustomName", "Representative_document", "Representative_Docs", "Top_n_words"])
dftexts["text_id"] = "id_" + dftexts.index.astype(str)
dftexts


In [ ]:
df_ia2 = df_ia1.merge(dftexts[["text_id","Topic","CustomName","Document"]], on =["text_id"], how = "left")
df_ia2